# 03. 경사 부여 (M2) — 노드 표고 + 엣지 경사

## 이 노트북이 하는 일
02의 차량 그래프에 **노드 표고(elev)** 를 붙이고, 그것으로 **엣지 경사(grade)** 를 방향별로 계산한다. 산복도로의 급경사를 그래프가 알게 만드는 단계.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 표고→경사인가:** 구급차 속도는 오르막/내리막에 따라 달라진다. 경사 = 두 노드의 표고차 ÷ 도로 길이.
- **왜 방향별(비대칭)인가:** 같은 도로도 오를 때와 내릴 때가 다르다. 그래서 `to_undirected` 금지 — 유향 그래프 유지.
- **왜 SRTM 30m(OpenTopoData)인가:** 국토지리정보원 2m DEM은 대용량 수동 다운로드가 필요. 우선 무료·키불필요 API로 **파이프라인을 완성**해두고, 나중에 2m DEM으로 표고 소스만 교체(로직 동일).
- **왜 배치 조회인가:** 노드가 4,800여 개라 하나씩 요청하면 느리고 제한에 걸린다. 100개씩 묶어 요청하고 초당 1회 제한을 지킨다.
- **왜 결측을 평균으로 채우나:** 바다·자료공백 좌표는 표고가 없을 수 있어, 경사 계산이 깨지지 않도록 전체 평균으로 임시 대체.

## 데이터 출처
- 표고: OpenTopoData 공개 API의 SRTM 30m(NASA). 최종본은 국토지리정보원 2m DEM(15059920)으로 교체 예정.
- 그래프: 02(OSM).

In [ ]:
%pip install pillow

In [ ]:
import os, math, warnings                 # math: 좌표→타일 변환 / warnings: 경고 숨김
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, requests    # numpy: 수치배열 / requests: 표고 API 호출
from io import BytesIO                         # (구 버전 잔재) 바이트 스트림
from PIL import Image                          # (구 버전 잔재) 이미지 처리
import osmnx as ox                             # 02 그래프 로드·저장
import geopandas as gpd                        # 좌표 변환
import networkx as nx                          # 그래프
import folium                                  # 경사 지도
print("osmnx", ox.__version__)
CRS_M = 5186                                   # 그래프 좌표계(평면, m)
os.makedirs("outputs", exist_ok=True)

## 1. M1 그래프 로드

In [ ]:
def _sf(x):                                                                           # 02가 grade/elev 자리를 빈 문자열("")로 저장 →
    try: return float(x)                                                              # 실수로 안전 변환(빈 값은 NaN)
    except: return float("nan")                                                       # 변환기 없이 로드하면 '' → float 변환 에러
G = ox.load_graphml("outputs/graph_drive_M1.graphml",                                 # 02가 저장한 그래프 읽기
    edge_dtypes={"grade":_sf,"grade_abs":_sf,"elev_u":_sf,"elev_v":_sf,"width_est":_sf},  # 빈 속성 안전 변환
    node_dtypes={"elev":_sf})
print("노드:", G.number_of_nodes(), "| 엣지:", G.number_of_edges(), "| CRS:", G.graph.get("crs"))
nodes4326 = ox.graph_to_gdfs(G, edges=False).to_crs(4326)                             # 표고 조회는 위경도가 필요 → 노드를 4326으로
lons = nodes4326.geometry.x.values                                                    # 각 노드의 경도 배열
lats = nodes4326.geometry.y.values                                                    # 각 노드의 위도 배열
ids  = list(nodes4326.index)                                                          # 노드 ID 순서(표고를 되붙일 때 사용)
print("표고 샘플링 대상 노드:", len(ids))

## 2. DEM 표고 샘플링 (OpenTopoData, SRTM 30m)
노드 위경도의 표고를 100개씩 묶어 조회. 공개 API 초당 1회 제한 준수.

In [ ]:
import time                                                                             # 요청 간 간격(제한 준수)용

LATLON = list(zip(lats, lons))                                                        # (위도,경도) 쌍 목록

def fetch_elev(latlon, dataset="srtm30m", chunk=100):                                 # 표고 배치 조회 함수
    out = []
    for i in range(0, len(latlon), chunk):                                            # 100개씩 끊어서
        part = latlon[i:i+chunk]
        locs = "|".join(f"{la:.6f},{lo:.6f}" for la, lo in part)                      # 'lat,lon|lat,lon|...' 형식으로 조립
        for attempt in range(4):                                                      # 실패 시 최대 4회 재시도
            try:
                r = requests.get(f"https://api.opentopodata.org/v1/{dataset}",        # SRTM 30m 데이터셋 호출
                                 params={"locations": locs}, timeout=30)
                if r.status_code == 200:                                              # 성공이면
                    out += [x["elevation"] for x in r.json()["results"]]              # 표고 값만 추출해 누적
                    break
                if r.status_code == 429:                                              # 요청 과다면
                    time.sleep(2); continue                                           # 잠깐 쉬고 재시도
                r.raise_for_status()
            except Exception:
                if attempt == 3:                                                      # 마지막 시도도 실패면 예외
                    raise
                time.sleep(2)
        time.sleep(1.1)                                                               # 공개 API 초당 1회 제한 준수
    return out

raw = fetch_elev(LATLON)                                                              # 전 노드 표고 조회(수십 초 소요)
elevs = np.array([e if e is not None else np.nan for e in raw], dtype=float)          # None→NaN 처리
if np.isnan(elevs).any():                                                             # 결측이 있으면
    elevs[np.isnan(elevs)] = np.nanmean(elevs)                                        # 전체 평균으로 임시 대체(경사계산 보호)
print("표고 샘플 수:", len(elevs))
print("표고 min/mean/max: %.1f / %.1f / %.1f m" % (elevs.min(), elevs.mean(), elevs.max()))

## 3. 노드에 표고, 엣지에 경사(방향별) 부여
grade = (도착노드표고 - 출발노드표고) / 길이 — 오르막 +, 내리막 -. to_undirected 금지.

In [ ]:
for nid, e in zip(ids, elevs):                                                        # 각 노드에 표고 기록
    G.nodes[nid]["elev"] = round(float(e), 2)

n_bad = 0                                                                             # 경사 계산 불가 엣지 수
for u, v, k, d in G.edges(keys=True, data=True):                                      # 모든 엣지 순회
    eu = G.nodes[u].get("elev"); ev = G.nodes[v].get("elev")                          # 시작/끝 노드 표고
    try:
        L = float(d.get("length", 0.0))                                              # 엣지 길이(m)
    except Exception:
        L = 0.0
    if eu is None or ev is None or L <= 0:                                            # 표고 없거나 길이 0이면
        d["elev_u"] = d["elev_v"] = d["grade"] = d["grade_abs"] = ""                  # 계산 불가로 비움
        n_bad += 1
        continue
    g = (ev - eu) / L                                                                # 경사 = 표고차 ÷ 길이 (방향 반영)
    d["elev_u"] = round(eu, 2); d["elev_v"] = round(ev, 2)                            # 양끝 표고 저장
    d["grade"] = round(g, 4); d["grade_abs"] = round(abs(g), 4)                       # 경사(부호)·경사 절대값 저장
print("경사 부여 완료. 계산 불가 엣지:", n_bad)

grades = pd.Series([d["grade_abs"] for *_, d in G.edges(keys=True, data=True)         # 경사 절대값 분포 요약
                    if isinstance(d.get("grade_abs"), float)])
print("\n|grade| 분포 (경사 절대값):")
print(grades.describe(percentiles=[.5,.75,.9,.95]).round(3).to_string())
for thr in [0.05, 0.10, 0.15, 0.20]:                                                  # 임계값별 급경사 비율
    print("  |grade| > %.2f : %5.1f%%" % (thr, (grades > thr).mean()*100))

## 4. 시각화 — 경사 색상 도로망\n초록(완만)→빨강(급경사).

In [ ]:
edges = ox.graph_to_gdfs(G, nodes=False).to_crs(4326)                                 # 엣지를 위경도로(지도용)
edges = edges[["geometry", "grade_abs"]].copy()                                       # 필요한 두 컬럼만
edges["grade_abs"] = pd.to_numeric(edges["grade_abs"], errors="coerce").fillna(0.0)   # 숫자화(빈 값은 0)

def grade_color(g):                                                                   # 경사에 따른 색
    if g > 0.15: return "#d7191c"                                                     # 15%↑ 빨강(급경사)
    if g > 0.10: return "#fdae61"                                                     # 10~15% 주황
    if g > 0.05: return "#ffffbf"                                                     # 5~10% 노랑
    return "#1a9641"                                                                  # 5%↓ 초록(완만)

m = folium.Map(location=[35.122, 129.045], zoom_start=14, tiles="cartodbpositron")    # 배경지도
folium.GeoJson(
    edges.to_json(),
    name="경사(grade)",
    style_function=lambda f: {"color": grade_color(f["properties"]["grade_abs"]),     # 엣지별 경사색
                              "weight": 2, "opacity": 0.8},
).add_to(m)

hosp = [("동아대학교병원",129.017604,35.120006),("부산대학교병원",129.019222,35.101054),  # 참고용 병원 마커
        ("인제대부산백병원",129.020572,35.146454)]
for nm, lo, la in hosp:
    folium.Marker([la, lo], tooltip=nm,
                  icon=folium.Icon(color="red", icon="plus", prefix="fa")).add_to(m)

legend = ('<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;'  # 범례 HTML
          'padding:8px 12px;border:1px solid #999;font-size:12px">'
          '<b>|경사|</b><br>'
          '<span style="color:#1a9641">&#9644;</span> &lt;5%<br>'
          '<span style="color:#ffd000">&#9644;</span> 5-10%<br>'
          '<span style="color:#fdae61">&#9644;</span> 10-15%<br>'
          '<span style="color:#d7191c">&#9644;</span> &gt;15%</div>')
m.get_root().html.add_child(folium.Element(legend))                                    # 범례를 지도에 삽입
folium.LayerControl().add_to(m)
m.save("outputs/grade_map_M2.html")                                                    # 경사 지도 저장
print("지도 저장: outputs/grade_map_M2.html")
m

## 5. 저장 (M2)

In [ ]:
ox.save_graphml(G, "outputs/graph_drive_M2.graphml")                                  # 표고·경사 포함 그래프 저장(다음 노트북용)
try:
    ox.save_graph_geopackage(G, "outputs/graph_drive_M2.gpkg")                        # QGIS용
except Exception as e:
    print("gpkg 경고:", e)

def stringify(gdf):                                                                   # parquet 저장용 타입 정리(02와 동일 이유)
    g = gdf.copy()
    for c in g.columns:
        if c == "geometry": continue
        if g[c].apply(lambda v: isinstance(v, (list, tuple))).any():
            g[c] = g[c].apply(lambda v: ";".join(map(str, v)) if isinstance(v, (list, tuple)) else v)
        if g[c].dtype == object:
            g[c] = g[c].astype(str)
    return g

ng, eg = ox.graph_to_gdfs(G)                                                          # 노드/엣지 표
for g, nm in [(ng, "nodes"), (eg, "edges")]:
    try: stringify(g).to_parquet(f"outputs/{nm}_drive_M2.parquet")                    # 분석용 parquet
    except Exception as e: print(nm, "parquet 경고:", e)
print("저장 완료:", [f for f in sorted(os.listdir("outputs")) if "M2" in f])